<a href="https://colab.research.google.com/github/Blenkzx/LINGUAGENS-DE-PROGRAMA-O/blob/main/SQLAlchemy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Níveis 1, 2 e 3: Básico, intermediário e Avançado

### Passo 1: Crie a conexão com um banco de dados local

Vamos começar criando a conexão com um banco de dados SQLite local chamado `sistema_rh.db`.

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

# Passo 1: Crie a conexão com um banco de dados local
engine = create_engine('sqlite:///sistema_rh.db')

print("Conexão com 'sistema_rh.db' criada com sucesso.")

Conexão com 'sistema_rh.db' criada com sucesso.


### Passo 2: Crie a tabela `funcionarios`

Agora, abriremos uma transação e utilizaremos SQL puro para criar a tabela `funcionarios` com as colunas `id`, `nome`, `cargo` e `salario`.

In [ ]:
# Passo 2: Abra uma transação e utilize a função text() para executar um comando CREATE TABLE
with engine.begin() as conn:
    create_table_sql = text("""
    CREATE TABLE IF NOT EXISTS funcionarios (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome VARCHAR(100) NOT NULL,
        cargo VARCHAR(100) NOT NULL,
        salario DECIMAL(10, 2) NOT NULL
    );
    """)
    conn.execute(create_table_sql)
    print("Tabela 'funcionarios' criada ou já existente.")

Tabela 'funcionarios' criada ou já existente.


### Passo 3 (Segurança): Simule a inserção de um novo funcionário

Vamos simular a inserção de um novo funcionário, demonstrando o uso de parâmetros seguros para prevenir ataques de injeção SQL.

**Pergunta reflexiva:** Por que nunca devemos concatenar strings diretamente no SQL (risco de injeção SQL) e como os placeholders resolvem isso?

**Resposta:** Concatenar strings diretamente no SQL é perigoso porque permite que usuários mal-intencionados injetem comandos SQL arbitrários na sua consulta (Injeção SQL). Isso pode levar ao roubo de dados, modificação ou exclusão de informações, ou até mesmo acesso não autorizado ao banco de dados. Os *placeholders* (como `:nome`, `:cargo`) resolvem esse problema tratando os valores de entrada como dados, e não como parte do código SQL. O banco de dados ou a biblioteca (como SQLAlchemy) garante que esses valores sejam devidamente escapados, impedindo que sejam interpretados como comandos executáveis.

In [ ]:
# Passo 3: Simule a inserção de um novo funcionário a partir de um formulário web
with engine.begin() as conn:
    insert_sql = text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)")

    # Dados do novo funcionário
    novo_funcionario = {
        'nome': 'João Silva',
        'cargo': 'Desenvolvedor Júnior',
        'salario': 4500.00
    }

    conn.execute(insert_sql, novo_funcionario)
    print(f"Funcionário '{novo_funcionario['nome']}' inserido com segurança.")

Funcionário 'João Silva' inserido com segurança.


### Passo 4: Valide a inserção consultando os dados

Para validar a inserção, consultaremos os dados da tabela `funcionarios` utilizando `pd.read_sql_query()`, que formatará o resultado diretamente como um DataFrame do Pandas.

In [ ]:
# Passo 4: Valide a inserção consultando os dados com pd.read_sql_query()
consulta_sql = text("SELECT id, nome, cargo, salario FROM funcionarios")
df_funcionarios = pd.read_sql_query(consulta_sql, engine)

print("Dados dos funcionários:")
display(df_funcionarios)

Dados dos funcionários:


,id,nome,cargo,salario
0,1,João Silva,Desenvolvedor Júnior,4500


\## Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)

O objetivo agora é abandonar o SQL em texto e usar as estruturas Python do SQLAlchemy Core, ideais para scripts de manipulação de dados e relatórios.

### Passo 1: Defina uma nova tabela chamada `projetos`

Vamos definir uma nova tabela chamada `projetos` de forma programática utilizando os objetos `Table`, `MetaData` e `Column` do SQLAlchemy Core. Em seguida, criaremos a tabela fisicamente no banco de dados.

In [ ]:
from sqlalchemy import Table, Column, Integer, String, MetaData, ForeignKey

# Passo 1: Defina uma nova tabela chamada projetos de forma programática
metadata_obj = MetaData()

# Reflete a tabela 'funcionarios' existente para que o MetaData a conheça
# Isso é crucial para que a ForeignKey possa referenciar a tabela 'funcionarios'
funcionarios = Table('funcionarios', metadata_obj, autoload_with=engine)

projetos = Table(
    'projetos',
    metadata_obj,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome_projeto', String(100), nullable=False),
    Column('descricao', String(255)),
    Column('id_gerente', Integer, ForeignKey(funcionarios.c.id)) # Corrigido: usando a referência explícita da coluna
)

# Crie a tabela fisicamente no banco (projetos e qualquer outra que metadata_obj conheça e não exista)
metadata_obj.create_all(engine)
print("Tabela 'projetos' criada ou já existente.")

Tabela 'projetos' criada ou já existente.


### Passo 2: Insira múltiplos projetos em lote

Receberemos uma lista de dicionários contendo múltiplos projetos e faremos uma inserção em lote (bulk insert) para otimizar a performance. O SQLAlchemy Core facilita isso com o comando `insert()` e passando a lista de dicionários.

In [ ]:
from sqlalchemy import insert

# Passo 2: Receba uma lista de dicionários e faça uma inserção em lote
lista_de_projetos = [
    {'nome_projeto': 'Desenvolvimento do App X', 'descricao': 'Criação de um novo aplicativo mobile', 'id_gerente': 1},
    {'nome_projeto': 'Otimização de Banco de Dados', 'descricao': 'Melhoria de performance do DB', 'id_gerente': 1},
    {'nome_projeto': 'Criação de Website Institucional', 'descricao': 'Site para a nova campanha', 'id_gerente': 1}
]

with engine.begin() as conn:
    conn.execute(insert(projetos), lista_de_projetos)
    print(f"{len(lista_de_projetos)} projetos inseridos em lote com sucesso.")

# Vamos verificar a inserção dos projetos
consulta_projetos_sql = text("SELECT id, nome_projeto, descricao, id_gerente FROM projetos")
df_projetos = pd.read_sql_query(consulta_projetos_sql, engine)

print("\nDados dos projetos:")
display(df_projetos)

3 projetos inseridos em lote com sucesso.

Dados dos projetos:


,id,nome_projeto,descricao,id_gerente
0,1,Desenvolvimento do App X,Criação de um novo aplicativo mobile,1
1,2,Otimização de Banco de Dados,Melhoria de performance do DB,1
2,3,Criação de Website Institucional,Site para a nova campanha,1


### Passo 3: Reajuste salarial dos 'Desenvolvedor Júnior'

A diretoria aprovou um reajuste. Utilizaremos a instrução `update()` do SQLAlchemy Core, combinada com `where()` e `values()`, para aumentar o salário apenas dos funcionários que ocupam o cargo de 'Desenvolvedor Júnior'.

In [ ]:
from sqlalchemy import update

# Passo 3: Utilize a instrução update para aumentar o salário de 'Desenvolvedor Júnior'

# Antes do reajuste
print("Salários antes do reajuste:")
df_antes_reajuste = pd.read_sql_query(text("SELECT nome, cargo, salario FROM funcionarios WHERE cargo = 'Desenvolvedor Júnior'"), engine)
display(df_antes_reajuste)

with engine.begin() as conn:
    # Define a tabela 'funcionarios' já criada no Nível 1 para usar na instrução update
    # É importante ter a Table object para usar com update, select, etc. do SQLAlchemy Core
    funcionarios = Table('funcionarios', metadata_obj, autoload_with=engine)

    stmt = update(funcionarios).where(funcionarios.c.cargo == 'Desenvolvedor Júnior').values(salario=funcionarios.c.salario * 1.10) # Aumento de 10%
    conn.execute(stmt)
    print("Salários dos 'Desenvolvedor Júnior' reajustados em 10%.")

# Após o reajuste
print("\nSalários após o reajuste:")
df_depois_reajuste = pd.read_sql_query(text("SELECT nome, cargo, salario FROM funcionarios WHERE cargo = 'Desenvolvedor Júnior'"), engine)
display(df_depois_reajuste)

Salários antes do reajuste:


,nome,cargo,salario
0,João Silva,Desenvolvedor Júnior,4950


Salários dos 'Desenvolvedor Júnior' reajustados em 10%.

Salários após o reajuste:


,nome,cargo,salario
0,João Silva,Desenvolvedor Júnior,5445


### Passo 4: Relatório salarial agregado por cargo

Para o relatório salarial, construiremos um `select` combinado com `func.avg()` para calcular a média salarial e `group_by()` para agrupar o resultado por cargo. Isso nos dará uma visão clara das médias salariais por categoria de funcionário.

### Passo 1: Transforme as tabelas em classes Python

Vamos utilizar `declarative_base()` para definir as classes `Departamento` e `FuncionarioORM`, mapeando as colunas das tabelas existentes para atributos de classe.

In [ ]:
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, Mapped, mapped_column
from sqlalchemy import Integer, String, DECIMAL, ForeignKey
from typing import List, Optional

# Passo 1: Transforme as tabelas em classes Python
Base = declarative_base()

class Departamento(Base):
    __tablename__ = 'departamentos' # Assumindo que criaremos uma nova tabela de departamentos
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(50), nullable=False, unique=True)

    # Relação com FuncionarioORM (definida no Passo 2)
    funcionarios: Mapped[List["FuncionarioORM"]] = relationship(back_populates="departamento", cascade="all, delete-orphan")

    def __repr__(self) -> str:
        return f"Departamento(id={self.id}, nome='{self.nome}')"

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios'
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(100), nullable=False)
    cargo: Mapped[str] = mapped_column(String(100), nullable=False)
    salario: Mapped[float] = mapped_column(DECIMAL(10, 2), nullable=False)

    # Chave estrangeira para Departamento
    departamento_id: Mapped[Optional[int]] = mapped_column(ForeignKey("departamentos.id"))

    # Relação com Departamento (definida no Passo 2)
    departamento: Mapped[Optional[Departamento]] = relationship(back_populates="funcionarios")

    def __repr__(self) -> str:
        return f"FuncionarioORM(id={self.id}, nome='{self.nome}', cargo='{self.cargo}', salario={self.salario})"

# Crie a tabela 'departamentos' e garanta que 'funcionarios' está mapeada
Base.metadata.create_all(engine)
print("Classes ORM definidas e tabelas criadas/mapeadas.")

### Passo 2: Estabeleça a relação entre as classes

Já configuramos a `ForeignKey` na tabela de funcionários (`departamento_id`) e utilizamos a função `relationship` em ambas as classes (`Departamento` e `FuncionarioORM`) para permitir a navegação de objeto para objeto. O código no Passo 1 já inclui essa configuração.

### Passo 3: Crie um departamento e adicione funcionários

Vamos criar uma fábrica de sessões com `sessionmaker(bind=engine)` e instanciar uma `Session`. Em seguida, criaremos um objeto da classe `Departamento` e adicionaremos funcionários a ele, persistindo tudo no banco de dados de uma só vez utilizando `sessao.add()` e `sessao.commit()`.

In [ ]:
# Passo 3: Crie uma fábrica de sessões e instancie uma Session
Session = sessionmaker(bind=engine)
sessao = Session()

try:
    # Crie um objeto Departamento
    departamento_ti = Departamento(nome='TI')

    # Adicione o departamento à sessão
    sessao.add(departamento_ti)

    # Crie funcionários e associe-os ao departamento
    # Reutilizando o funcionário 'João Silva' e adicionando outro

    # Primeiro, vamos atualizar o departamento_id do João Silva existente via ORM
    joao_silva = sessao.query(FuncionarioORM).filter_by(nome='João Silva').first()
    if joao_silva:
        joao_silva.departamento = departamento_ti
        print(f"Funcionário '{joao_silva.nome}' associado ao departamento de '{departamento_ti.nome}'.")

    # Adicionando um novo funcionário diretamente via ORM
    novo_func = FuncionarioORM(nome='Maria Oliveira', cargo='Analista de Testes', salario=6000.00, departamento=departamento_ti)
    sessao.add(novo_func)

    # Persista todos no banco de dados
    sessao.commit()
    print("Departamento e funcionários associados persistidos com sucesso.")

except Exception as e:
    sessao.rollback()
    print(f"Ocorreu um erro: {e}")
finally:
    sessao.close()

### Passo 4: Faça uma consulta orientada a objetos

Vamos fazer uma consulta orientada a objetos para listar todos os funcionários do departamento de "TI". Utilizaremos `sessao.execute(select(...)).scalars().all()` para que o SQLAlchemy retorne os objetos instanciados da classe `FuncionarioORM` em vez de linhas textuais. No final, utilizaremos `sessao.close()` para fechar a sessão adequadamente.

In [ ]:
from sqlalchemy import select

# Passo 4: Faça uma consulta orientada a objetos para listar todos os funcionários do departamento de "TI"
Session = sessionmaker(bind=engine)
sessao = Session()

try:
    # Consulta para encontrar o departamento de TI
    departamento_ti = sessao.query(Departamento).filter_by(nome='TI').first()

    if departamento_ti:
        print(f"\nFuncionários do departamento de '{departamento_ti.nome}':")
        # Consulta os funcionários associados a este departamento
        for funcionario in departamento_ti.funcionarios:
            print(f"  - {funcionario.nome} ({funcionario.cargo}, Salário: {funcionario.salario:.2f})")
    else:
        print("Departamento 'TI' não encontrado.")

    # Outra forma de consultar: diretamente pela classe FuncionarioORM com filtro
    print("\nConsultando via FuncionarioORM para 'TI':")
    funcionarios_ti_direto = sessao.execute(
        select(FuncionarioORM).
        join(Departamento).
        where(Departamento.nome == 'TI')
    ).scalars().all()

    for funcionario in funcionarios_ti_direto:
        print(f"  - {funcionario.nome} ({funcionario.cargo}, Salário: {funcionario.salario:.2f})")

except Exception as e:
    print(f"Ocorreu um erro durante a consulta: {e}")
finally:
    sessao.close()
    print("Sessão fechada adequadamente.")